In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# FIXED VERSION (same intent; syntax + parentheses + casts)
# Key fixes:
# - Auditorium_Experience_ID cast to string before isin
# - Member/Companion_Priced_CC_Revenue: fixed broken parentheses + alias placement
# - Companion redemption: fixed missing parenthesis in when()
# - Kept your original aggregations + added the two new revenue fields
# ============================================================

# Tables
T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_date  = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_sess  = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_session")
T_films = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
T_perf = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_performance_type")

# ------------------------------------------------------------
# 0) Session format flags (priority order) -> Format
# ------------------------------------------------------------
session_format_flags = [
    "IS_3DFlag","IS_4DxFlag","Is_70mmflag","Is_AVXflag","Is_ScreenXflag",
    "Is_DBoxflag","Is_Dolby_Atmosflag","Is_DSflag","Is_HFRflag","Is_IMAXflag",
    "Is_IMAX_VRflag","Is_Non_IMAXflag","Is_Clubhouseflag",
]

format_expr = F.coalesce(
    *[F.when(F.col(f"s.{c}") == "Yes", F.lit(c)) for c in session_format_flags],
    F.lit("Regular")
)

# ------------------------------------------------------------
# 1) Base joined + filtered (single scan)
# ------------------------------------------------------------
df_base = (
    T_trans.alias("t")
    .join(T_date.alias("d"), F.col("t.VisitDateId") == F.col("d.dateid"), "inner")
    .join(T_sess.alias("s"), F.col("t.SessionSkey") == F.col("s.sessionskey"), "left")
    .join(T_perf.alias("p"), F.col("s.Performance_Type_ID") == F.col("p.performance_type_id"), "left")
    .join(T_films.alias("f"), F.col("t.film_id") == F.col("f.film_id"), "left")
    .filter(F.col("d.datefull").between(F.lit("2025-01-01"), F.lit("2025-12-31")))
    .filter(F.col("s.Auditorium_Experience_ID").cast("string").isin("5", "6"))

    .withColumn("Format", format_expr)
)

# ------------------------------------------------------------
# 2) Event cinema rollup at (title, film_id, format, location)
# ------------------------------------------------------------
df_event_cinema = (
    df_base
    .filter(F.col("t.TicketFLAG") == "Yes")
    .groupBy(
        F.col("p.performance_type"),
        F.col("f.title_name").alias("title_name"),
        F.col("f.film_id").alias("film_id"),
        F.col("Format"),
        F.col("t.LocationId").alias("LocationId")
    )
    .agg(
        F.round(F.sum(F.coalesce(F.col("t.Quantity"), F.lit(0))), 2).alias("Total_Tickets_Sold"),
        F.round(F.sum(F.coalesce(F.col("t.Net_Revenue"), F.lit(0.0))), 2).alias("Total_Ticket_Revenue"),

        # NEW: CC ticket revenue broken out by discount type (using Net_Revenue on ticket rows)
        F.round(
            F.sum(
                F.when(
                    (F.col("t.CineClub_Discount_Type_ID").cast("string").isin("5","6")),
                    F.coalesce(F.col("t.Net_Revenue"), F.lit(0.0))
                ).otherwise(F.lit(0.0))
            ),
            2
        ).alias("CC_Ticket_Revenue"),

        F.sum(
            F.when(
                F.col("t.CineClub_Discount_Type_ID").cast("string") == "5",
                F.coalesce(F.col("t.Quantity"), F.lit(0))
            ).otherwise(F.lit(0))
        ).alias("Total_CineClub_Redemptions"),

        F.sum(
            F.when(
                F.col("t.CineClub_Discount_Type_ID").cast("string") == "6",
                F.coalesce(F.col("t.Quantity"), F.lit(0))
            ).otherwise(F.lit(0))
        ).alias("Total_CineClub_Companion_Redemptions"),
    )
    .withColumn(
        "Total_CineClub_Portion",
        F.when(
            F.col("Total_Tickets_Sold") > 0,
            (F.col("Total_CineClub_Redemptions") + F.col("Total_CineClub_Companion_Redemptions")) / F.col("Total_Tickets_Sold")
        ).otherwise(F.lit(None))
    )
)

# ------------------------------------------------------------
# 3) Mode ticket price
# ------------------------------------------------------------
df_price = (
    df_base
    .select(
        F.col("t.film_id").alias("film_id"),
        F.col("t.LocationId").alias("LocationId"),
        F.col("t.VisitDateId").alias("VisitDateId"),
        F.col("t.TicketFLAG").alias("TicketFLAG"),
        F.col("t.Quantity").alias("Quantity"),
        F.col("t.Net_Revenue").alias("Net_Revenue"),
    )
    .filter(F.col("TicketFLAG") == "Yes")
    .filter(F.col("VisitDateId").cast("string").startswith("2025"))
    .filter(F.col("Quantity").isNotNull() & (F.col("Quantity") > 0))
    .filter(F.col("Net_Revenue").isNotNull())
    .withColumn("unit_price", (F.col("Net_Revenue") / F.col("Quantity")).cast("double"))
    .withColumn("unit_price_r", F.round(F.col("unit_price"), 2))
)

df_counts = (
    df_price
    .groupBy("film_id", "LocationId", "unit_price_r")
    .agg(F.count("*").alias("cnt"))
)

w_mode = Window.partitionBy("film_id", "LocationId").orderBy(F.col("cnt").desc(), F.col("unit_price_r").desc())

df_mode_price = (
    df_counts
    .withColumn("rn", F.row_number().over(w_mode))
    .filter(F.col("rn") == 1)
    .select("film_id", "LocationId", F.col("unit_price_r").alias("Mode_Ticket_Price"), "cnt")
)

# ------------------------------------------------------------
# 4) Final title-level rollup
# ------------------------------------------------------------
df_final = (
    df_event_cinema
    .join(df_mode_price, ["film_id", "LocationId"], "inner")
    .drop("cnt")
    .groupBy( F.col("p.performance_type"), "title_name")
    .agg(
        F.sum("Total_Tickets_Sold").alias("Total_Tickets_Sold"),
        F.sum("Total_Ticket_Revenue").alias("Total_Ticket_Revenue"),
        F.sum("CC_Ticket_Revenue").alias("CC_Ticket_Revenue"),
        F.sum("Total_CineClub_Redemptions").alias("Total_CineClub_Redemptions"),
        F.sum("Total_CineClub_Companion_Redemptions").alias("Total_CineClub_Companion_Redemptions"),
        F.max("Mode_Ticket_Price").alias("Mode_Ticket_Price")
    )
    .withColumn(
        "Total_CineClub_Portion",
        F.when(
            F.col("Total_Tickets_Sold") > 0,
            (F.col("Total_CineClub_Redemptions") + F.col("Total_CineClub_Companion_Redemptions")) / F.col("Total_Tickets_Sold")
        ).otherwise(F.lit(None))
    )
    .withColumn(
        "CineClub_Discount",
        F.round(F.col("Mode_Ticket_Price") - 9.99) *
        (F.col("Total_CineClub_Redemptions") + F.col("Total_CineClub_Companion_Redemptions"))
    )
    .orderBy(F.col("Total_Ticket_Revenue").desc())
)

# ------------------------------------------------------------
# 5) Concession rollup (title-level)
# ------------------------------------------------------------
df_concessions = (
    df_base
    .groupBy( F.col("p.performance_type"),F.col("f.title_name").alias("title_name"))
    .agg(
        F.round(
            F.sum(F.when(F.col("t.TicketFLAG") == "Yes", F.coalesce(F.col("t.Quantity"), F.lit(0))).otherwise(F.lit(0))),
            2
        ).alias("Total_Tickets_Sold"),
        F.round(
            F.sum(F.when(F.col("t.TicketFLAG") == "Yes", F.coalesce(F.col("t.Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))),
            2
        ).alias("Total_Ticket_Revenue"),
        F.sum(
            F.when(
                (F.col("t.TicketFLAG") == "Yes") & (F.col("t.CineClub_Discount_Type_ID").cast("string") == "5"),
                F.coalesce(F.col("t.Quantity"), F.lit(0))
            ).otherwise(F.lit(0))
        ).alias("Total_CineClub_Redemptions"),
        F.sum(
            F.when(
                (F.col("t.TicketFLAG") == "Yes") & (F.col("t.CineClub_Discount_Type_ID").cast("string") == "6"),
                F.coalesce(F.col("t.Quantity"), F.lit(0))
            ).otherwise(F.lit(0))
        ).alias("Total_CineClub_Companion_Redemptions"),
        F.round(
            F.sum(F.when(F.col("t.ConcessionFLAG") == "Yes", F.coalesce(F.col("t.Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))),
            2
        ).alias("Total_Concession_Revenue"),
        F.round(
            F.sum(
                F.when(
                    (F.col("t.ConcessionFLAG") == "Yes") & (F.col("t.CineClubFLAG") == "Yes"),
                    F.coalesce(F.col("t.Net_Revenue"), F.lit(0.0))
                ).otherwise(F.lit(0.0))
            ),
            2
        ).alias("CineClub_Concession_Revenue"),
    )
    .withColumn(
        "CineClub_Concession_Share",
        F.when(
            F.col("Total_Concession_Revenue") > 0,
            F.col("CineClub_Concession_Revenue") / F.col("Total_Concession_Revenue")
        ).otherwise(F.lit(None))
    )
    .orderBy(F.col("Total_Ticket_Revenue").desc())
)

# Displays
display(df_event_cinema)
display(df_mode_price)
display(df_final)
display(df_concessions)


In [0]:
from pyspark.sql import functions as F

# df_final is already title-level
# df_concessions is title-level (has concession revenue + shares)

df_final_merged = (
    df_final.alias("a")
    .join(
        df_concessions.select(
            "performance_type",
            "title_name",
            "Total_Concession_Revenue",
            "CineClub_Concession_Revenue",
            "CineClub_Concession_Share"
        ).alias("c"),
        on="title_name",
        how="left"
    )
    # (optional) recompute concession share safely in case of nulls
    .withColumn(
        "CineClub_Concession_Share",
        F.when(
            F.col("c.Total_Concession_Revenue").isNotNull() & (F.col("c.Total_Concession_Revenue") > 0),
            F.col("c.CineClub_Concession_Revenue") / F.col("c.Total_Concession_Revenue")
        ).otherwise(F.col("c.CineClub_Concession_Share"))
    )
    .select(
        "a.performance_type",
        F.col("a.title_name"),
        F.col("a.Total_Tickets_Sold"),
        F.col("a.Total_Ticket_Revenue"),
        F.col("a.CC_Ticket_Revenue"),
        F.col("a.Total_CineClub_Redemptions"),
        F.col("a.Total_CineClub_Companion_Redemptions"),
        F.col("a.Total_CineClub_Portion"),
        F.col("a.Mode_Ticket_Price"),
        F.col("a.CineClub_Discount"),
        F.col("c.Total_Concession_Revenue"),
        F.col("c.CineClub_Concession_Revenue"),
        F.col("CineClub_Concession_Share")
    )
    .orderBy(F.col("Total_Ticket_Revenue").desc())
)

display(df_final_merged)
